In [14]:
import sys, tensorflow as tf
from PIL import Image
import numpy as np
import keras
import os
from keras.applications import MobileNetV2

In [15]:
data_dir = "/Users/ismail/Documents/GitHub/waste-reducing-app-hackathon/model-creation/images"
test_image = "/Users/ismail/Documents/GitHub/waste-reducing-app-hackathon/model-creation/test.png"

print("Executable:", sys.executable)
print("TF Version:", tf.__version__)
print(tf.config.list_physical_devices('GPU'))

Executable: /Users/ismail/Documents/GitHub/waste-reducing-app-hackathon/model-creation/.venv/bin/python
TF Version: 2.18.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
converted_count = 0
deleted_count = 0

for dir in os.listdir(data_dir):
    if dir == ".DS_Store":
        continue
    for file in os.listdir(os.path.join(data_dir, dir)):
        if file == ".DS_Store":
            continue
        allowed_exts = ["jpeg", "png", "gif", "bmp"]
        try:
            with Image.open(os.path.join(data_dir, dir, file)) as image_loaded:
                image_loaded.verify()

            with Image.open(os.path.join(data_dir, dir, file)) as image_loaded:
                acutal_ext = image_loaded.format.lower()
                if acutal_ext not in allowed_exts:
                    image_loaded.save(
                        os.path.join(data_dir, dir, file.split(".")[0] + ".png"),
                        "PNG",
                        icc_profile=None,
                    )
                    os.remove(os.path.join(data_dir, dir, file))
                    converted_count += 1
                    print(f"Converted {file} to PNG")
                    file = os.path.join(data_dir, dir, file.split(".")[0] + ".png")
                    if image_loaded.mode in ("RGBA", "P", "CMYK"):
                        image_loaded = image_loaded.convert("RGB")
                else:
                    image_loaded.save(
                        os.path.join(data_dir, dir, file),
                        image_loaded.format,
                        icc_profile=None,
                    )
        except Exception as e:
            os.remove(os.path.join(data_dir, dir, file))
            deleted_count += 1
            print(f"Deleted {file} due to error")

print(f"Converted: {converted_count}")
print(f"Deleted: {deleted_count}")

Converted: 0
Deleted: 0


In [4]:
data_augmentation = keras.Sequential(
    [
        keras.layers.RandomFlip("horizontal"),
        keras.layers.RandomRotation(0.1),
        keras.layers.RandomZoom(0.2),
        keras.layers.RandomFlip("vertical"),
        keras.layers.RandomContrast(0.2),
        keras.layers.RandomTranslation(0.1, 0.1)
    ]
)

2026-11-13 21:16:52.094390: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M5
2026-11-13 21:16:52.094409: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 32.00 GB
2026-11-13 21:16:52.094414: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 12.48 GB
I0000 00:00:1794626212.094423  501357 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1794626212.094437  501357 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [5]:
train_ds = keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    image_size=(224, 224),
    batch_size=32,
    seed=42,
)

val_ds = keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    image_size=(224, 224),
    batch_size=32,
    seed=42,
)

class_names = train_ds.class_names
num_classes = len(class_names)

Found 2791 files belonging to 17 classes.
Using 2233 files for training.
Found 2791 files belonging to 17 classes.
Using 558 files for validation.


In [6]:
base_model = MobileNetV2(
    input_shape=(224, 224, 3), include_top=False, weights="imagenet"
)

base_model.trainable = False

In [7]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

In [8]:
model = keras.Sequential(
    [
        data_augmentation,
        keras.layers.Lambda( 
            lambda x: keras.applications.mobilenet_v2.preprocess_input(x)
        ),
        base_model,
        keras.layers.GlobalAveragePooling2D(),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(num_classes, activation="softmax"),
    ]
)

In [13]:
model.compile(
    loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"]
)

model.fit(train_ds, validation_data=val_ds, epochs=15)

Epoch 1/15
70/70 ━━━━━━━━━━━━━━━━━━━━ 33s 378ms/step - accuracy: 0.6659 - loss: 1.1545 - val_accuracy: 0.2330 - val_loss: 12.5203
Epoch 2/15
70/70 ━━━━━━━━━━━━━━━━━━━━ 22s 318ms/step - accuracy: 0.6713 - loss: 1.0974 - val_accuracy: 0.4032 - val_loss: 7.9437
Epoch 3/15
70/70 ━━━━━━━━━━━━━━━━━━━━ 23s 322ms/step - accuracy: 0.7277 - loss: 0.8745 - val_accuracy: 0.2312 - val_loss: 15.3466
Epoch 4/15
70/70 ━━━━━━━━━━━━━━━━━━━━ 22s 316ms/step - accuracy: 0.7300 - loss: 0.8419 - val_accuracy: 0.2240 - val_loss: 14.1611
Epoch 5/15
70/70 ━━━━━━━━━━━━━━━━━━━━ 24s 340ms/step - accuracy: 0.7524 - loss: 0.7845 - val_accuracy: 0.2007 - val_loss: 23.8259
Epoch 6/15
70/70 ━━━━━━━━━━━━━━━━━━━━ 24s 336ms/step - accuracy: 0.7828 - loss: 0.6753 - val_accuracy: 0.1326 - val_loss: 18.9537
Epoch 7/15
70/70 ━━━━━━━━━━━━━━━━━━━━ 24s 335ms/step - accuracy: 0.7940 - loss: 0.6533 - val_accuracy: 0.2563 - val_loss: 10.1922
Epoch 8/15
70/70 ━━━━━━━━━━━━━━━━━━━━ 25s 348ms/step - accuracy: 0.8164 - loss: 0.5522 - va

KeyboardInterrupt: 

In [10]:
base_model.trainable = True

for layer in base_model.layers[:100]:
    layer.trainable = False

In [11]:
custom_omtimizer = keras.optimizers.Adam(learning_rate=1e-4)

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=custom_omtimizer,
    metrics=["accuracy"]
)

In [12]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=40,
    initial_epoch=15,
)

model.save("waste_classification_model.keras")

Epoch 16/40
70/70 ━━━━━━━━━━━━━━━━━━━━ 35s 389ms/step - accuracy: 0.6274 - loss: 1.1972 - val_accuracy: 0.6165 - val_loss: 1.3293
Epoch 17/40
70/70 ━━━━━━━━━━━━━━━━━━━━ 26s 373ms/step - accuracy: 0.7058 - loss: 0.9407 - val_accuracy: 0.6147 - val_loss: 1.3987
Epoch 18/40
70/70 ━━━━━━━━━━━━━━━━━━━━ 26s 373ms/step - accuracy: 0.7456 - loss: 0.8057 - val_accuracy: 0.6201 - val_loss: 1.4213
Epoch 19/40
70/70 ━━━━━━━━━━━━━━━━━━━━ 26s 373ms/step - accuracy: 0.7882 - loss: 0.6783 - val_accuracy: 0.6344 - val_loss: 1.4144
Epoch 20/40
70/70 ━━━━━━━━━━━━━━━━━━━━ 29s 404ms/step - accuracy: 0.8124 - loss: 0.6124 - val_accuracy: 0.6326 - val_loss: 1.4620
Epoch 21/40
70/70 ━━━━━━━━━━━━━━━━━━━━ 27s 386ms/step - accuracy: 0.8433 - loss: 0.5109 - val_accuracy: 0.6272 - val_loss: 1.4475
Epoch 22/40
70/70 ━━━━━━━━━━━━━━━━━━━━ 28s 403ms/step - accuracy: 0.8580 - loss: 0.4653 - val_accuracy: 0.6416 - val_loss: 1.4452
Epoch 23/40
70/70 ━━━━━━━━━━━━━━━━━━━━ 27s 385ms/step - accuracy: 0.8607 - loss: 0.4147 - 